# Deepfake Detection - Train All Strategies (c23)
Make sure to:
- Set **Accelerator → P100** in notebook settings
- Add **both** datasets via **+ Add Data**:
  - `katroue/ff-c23-frames-1` (original, Deepfakes, Face2Face)
  - `katroue/ff-c23-frames-2` (FaceSwap, NeuralTextures, splits)

In [ ]:
# Step 1: Clone repo (shallow clone = latest commit only, fast)
import os
if not os.path.exists('deepfake_project_comp6341'):
    !git clone --depth 1 --branch training https://github.com/katherinedemers/deepfake_project_comp6341.git
%cd deepfake_project_comp6341
!git fetch origin training && git checkout training && git pull origin training

In [ ]:
# Step 2: Install missing dependencies (torch, torchvision, numpy, sklearn already on Kaggle)
!pip install timm grad-cam -q

In [ ]:
# Step 3: Map both Kaggle datasets to expected directory structure
import os

DATASET1 = '/kaggle/input/datasets/katroue/ff-c23-frames-1'  # original, Deepfakes, Face2Face
DATASET2 = '/kaggle/input/datasets/katroue/ff-c23-frames-2'  # FaceSwap, NeuralTextures, splits
DATA_ROOT = '/kaggle/working/deepfake_project_comp6341/data'

dirs = [
    f'{DATA_ROOT}/original_sequences/youtube/c23',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

links = {
    f'{DATA_ROOT}/original_sequences/youtube/c23/images':           f'{DATASET1}/original',
    f'{DATA_ROOT}/manipulated_sequences/Deepfakes/c23/images':      f'{DATASET1}/Deepfakes',
    f'{DATA_ROOT}/manipulated_sequences/Face2Face/c23/images':      f'{DATASET1}/Face2Face',
    f'{DATA_ROOT}/manipulated_sequences/FaceSwap/c23/images':       f'{DATASET2}/FaceSwap',
    f'{DATA_ROOT}/manipulated_sequences/NeuralTextures/c23/images': f'{DATASET2}/NeuralTextures',
    f'{DATA_ROOT}/splits':                                           f'{DATASET2}/splits',
}
for link, target in links.items():
    if os.path.islink(link) and not os.path.exists(link):
        os.unlink(link)  # remove broken symlink
    if not os.path.lexists(link):
        os.symlink(target, link)

# Verify
for link in links:
    count = len(os.listdir(link))
    print(f'{link.split("/data/")[1]}: {count} entries')

In [ ]:
# Step 4: Update configs to point to Kaggle paths
import glob, re

DATA_ROOT = '/kaggle/working/deepfake_project_comp6341/data/'
for cfg in glob.glob('configs/c23/*.yaml'):
    with open(cfg) as f:
        content = f.read()
    content = re.sub(r'data_root:.*', f'data_root: "{DATA_ROOT}"', content)
    content = re.sub(r'save_dir: "results/', 'save_dir: "/kaggle/working/results/', content)
    content = re.sub(r'log_dir: "results/', 'log_dir: "/kaggle/working/results/', content)
    with open(cfg, 'w') as f:
        f.write(content)
    print(f'Updated {cfg}')

In [ ]:
# Step 5: Run all 6 strategies
!bash scripts/train_all_strategies_c23.sh

In [ ]:
# Step 6: Copy results to Kaggle output for download
import shutil
shutil.copytree('/kaggle/working/results', '/kaggle/output/results', dirs_exist_ok=True)
print('Results saved to /kaggle/output/results')
!find /kaggle/output/results/models -name 'best_model.pth'